In [9]:
import sys
import os

print("Python Executable:", sys.executable)
print("Operating System :", os.name, sys.platform)

Python Executable: /opt/conda/bin/python
Operating System : posix linux


## PENJELASAN ## 
> Method sys.executable dan os.name, untuk membantu mengetahui codingan ini berjalan menggunakan kernel apa dan Operating System apa, dan di outputnya tertulis untuk OS nya menggunakan posix linux yang merupakan hasil dari menggunakan Docker

### 01 - DATA PROFILING ###

In [10]:
import polars as pl
import duckdb

DATA_PATH = "/workspace/data/raw/fire_archive_SV-C2_812847.csv"

lf = pl.scan_csv(DATA_PATH)
lf.collect_schema()

Schema([('latitude', Float64),
        ('longitude', Float64),
        ('brightness', Float64),
        ('scan', Float64),
        ('track', Float64),
        ('acq_date', String),
        ('acq_time', Int64),
        ('satellite', String),
        ('instrument', String),
        ('confidence', String),
        ('version', Int64),
        ('bright_t31', Float64),
        ('frp', Float64),
        ('daynight', String),
        ('type', Int64)])

## Penjelasan :
1. Mengimport library polars dan duckdb, dipilih keduanya karena keduanya memiliki kelebihan dan kemampuan teknis yang lebih efisien dalam komputasi juga dalam penyimpanan. Selain itu, menggunakan konsep Lazy Scan, dimana tidak semua akan di load ke memori, tetapi fokus pada data yang "memang dibutuhkan" yang akan terproses.
2. lf = didefinisikan untuk menjadi variabel yang menampung dataset csv nya
3. pl.scan = untuk membaca csv yang sesuai dengan alamatnya
4. lf.collect_schema = untuk menampilkan isi dalam datasetnya berupa fitur fiturnya atau kolom kolomnya beserta tipe data nya per masing masing kolom

## 1.1 Ukuran Dataset ##

In [11]:
n_rows = lf.select(pl.len()).collect().item()
n_cols = len(lf.collect_schema())
print(f"Number of rows: {n_rows}")
print(f"Number of columns: {n_cols}")


Number of rows: 1535566
Number of columns: 15


## PENJELASAN ##
1. n_rows untuk mendefinisikan isi dataset yang berbentuk baris, dengan method "pl.len" untuk menampilkan berapa jumlah baris secara total, outputnya sebanyak 1.535.566 juta baris
2. n_cols untuk mendefinisikan isi dataset yang berbentuk kolom, lalu menggunakan len dan memanggil fungsi lf.collect_schema yang sebelumnya sudah dipakai, outputnya sebanyak 15 kolom

## 1.2 Deskripsi Statistik Dataset ##

In [12]:
con = duckdb.connect()
summary = con.execute(f"""
SUMMARIZE SELECT * FROM read_csv_auto('{DATA_PATH}')
""").pl()
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""latitude""","""DOUBLE""","""-10.97676""","""5.87273""",840522,"""-3.304907118170085""","""3.7413716323788995""","""-7.301347916581497""","""-2.713409025850735""","""-0.3771967606357092""",1535566,0.00
"""longitude""","""DOUBLE""","""95.01992""","""140.9774""",1126093,"""114.7826630891861""","""10.166162731722366""","""106.9837823090027""","""113.85024061494195""","""120.55012836023539""",1535566,0.00
"""brightness""","""DOUBLE""","""208.0""","""367.0""",6175,"""331.95654492219927""","""15.992048003287552""","""326.5240802685811""","""334.35653405168586""","""341.471533731329""",1535566,0.00
"""scan""","""DOUBLE""","""0.32""","""0.8""",47,"""0.4547046821822925""","""0.08199858940544344""","""0.39""","""0.4302234957154089""","""0.4999920679041167""",1535566,0.00
"""track""","""DOUBLE""","""0.36""","""0.78""",43,"""0.46913609704829323""","""0.10876248521372833""","""0.37741845278349295""","""0.43194039399123707""","""0.5464417899657208""",1535566,0.00
…,…,…,…,…,…,…,…,…,…,…,…
"""version""","""BIGINT""","""2""","""2""",1,"""2.0""","""0.0""","""2""","""2""","""2""",1535566,0.00
"""bright_t31""","""DOUBLE""","""150.0""","""380.61""",7908,"""294.21808263532785""","""8.64783439973069""","""289.73095212403945""","""294.3989078361154""","""299.90170052248425""",1535566,0.00
"""frp""","""DOUBLE""","""0.0""","""1322.99""",10172,"""7.734733114695157""","""13.205919662525844""","""2.488781218354277""","""4.615216925111091""","""8.238467365234193""",1535566,0.00


## PENJELASAN ##
1. Menggunakan DuckDB dengan menyertakan SUMMARIZE untuk memberi statistik deskriptif per kolom yang meliputi nilai min, max, approx_unique, null% hingga null percentage dalam satu query SQL.
2. Penggunaan DuckDB mempercepat dalam eksplorasi ad-hoc dibanding menulis banyak agregasi manual di Polars.
3. Lalu pada nilai approx_unique, terdapat kolom yang memiliki nilai hanya 1, yaitu version. Hal tersebut bisa terjadi karena melihat dari file yang diunduh dan file tersebut memiliki versinya dari sumbernya. Jadi file tersebut merupakan satu dari banyaknya file yang tersedia dan yang diberikan oleh sumbernya.

## 1.3 Deskripsi Rentang Temporan dan Spasial ##

In [13]:
bounds = lf.select(
    pl.col("acq_date").min().alias("min_acq_date"),
    pl.col("acq_date").max().alias("max_acq_date"),
    pl.col("latitude").min().alias("min_latitude"),
    pl.col("latitude").max().alias("max_latitude"),
    pl.col("longitude").min().alias("min_longitude"),
    pl.col("longitude").max().alias("max_longitude")
).collect()
bounds

min_acq_date,max_acq_date,min_latitude,max_latitude,min_longitude,max_longitude
str,str,f64,f64,f64,f64
"""2016-01-01""","""2026-06-30""",-10.97676,5.87273,95.01992,140.9774


## PENJELASAN ##
1. Pada cell ini, fokus pada rentang temporal (keterangan waktu) dan rentang spasial (tempat yang didefinisikan menggunakan titik koordinat)
2. Pada outputnya, menghasilkan nilai min dan max tiap kolom yang berjenis temporal dan spasial.

## 1.4 Distribusi Kolom Kategorikal Kunci ##

In [14]:
for col in ["confidence", "type", "daynight", "satellite"]:
    print(f"Value counts for column '{col}':")
    print (
        lf.group_by(col)
        .agg(pl.len().alias("count"))
        .with_columns((pl.col("count") / n_rows * 100).round(2).alias("percentage"))
        .sort("count", descending=True)
        .collect()
    )
    print("\n")

Value counts for column 'confidence':
shape: (3, 3)
┌────────────┬─────────┬────────────┐
│ confidence ┆ count   ┆ percentage │
│ ---        ┆ ---     ┆ ---        │
│ str        ┆ u32     ┆ f64        │
╞════════════╪═════════╪════════════╡
│ n          ┆ 1386873 ┆ 90.32      │
│ l          ┆ 101396  ┆ 6.6        │
│ h          ┆ 47297   ┆ 3.08       │
└────────────┴─────────┴────────────┘


Value counts for column 'type':
shape: (4, 3)
┌──────┬─────────┬────────────┐
│ type ┆ count   ┆ percentage │
│ ---  ┆ ---     ┆ ---        │
│ i64  ┆ u32     ┆ f64        │
╞══════╪═════════╪════════════╡
│ 0    ┆ 1416095 ┆ 92.22      │
│ 2    ┆ 79308   ┆ 5.16       │
│ 1    ┆ 32173   ┆ 2.1        │
│ 3    ┆ 7990    ┆ 0.52       │
└──────┴─────────┴────────────┘


Value counts for column 'daynight':
shape: (2, 3)
┌──────────┬─────────┬────────────┐
│ daynight ┆ count   ┆ percentage │
│ ---      ┆ ---     ┆ ---        │
│ str      ┆ u32     ┆ f64        │
╞══════════╪═════════╪════════════╡
│ D   

## PENJELASAN ##
1. Hasil output dalam cell ini fokus pada kolom kunci yang terdiri dari confidence, type, daynight, satellite
2. Confidence = tolak ukur seberapa yakin satelit mendeteksi bahwa suatu titik itu benar benar ada aktivitas api, dengan rincian type, n = nominal, l = low dan h = high. Jika melihat hasilnya, 90 sekian persen itu memiliki dataset sebagian titik api nya benar benar terdeteksi dalam keadaan normal walaupun masih terdapat gangguan kecil. Sementara untuk low nya, sebanyak 6 sekian persen yang berarti angka tersebut titik api nya belum terdeteksi dengan baik karena terdapat gangguan secara alam, entah dari pantulan laut, asap, awan dan sebagainya. Selain itu, untuk high hanya dapat 3.08 persen yang berarti hasil deteksi sebanyak angka tersebut titiknya benar benar terdeteksi tanpa ada gangguan apapun.
3. Type = untuk menjelaskan terkait jenis sebab munculnya aktivitas api yang dibagi menjadi 4 tipe. 0 sebagai vegetation fire, mengartikan bahwa titik api itu muncul karena kebakaran hutan. 1 sebagai active volcano, mengartikan bahwa titik api itu muncul karena dari gunung berapi. 2 sebagai other static land source, mengartikan bahwa titik api itu muncul dari buatan manusia, seperti pabrik, kawasan industrial yang memunculkan titik api atau sumber panas yang bukan alami. 3 sebagai off shore, untuk titik api yang muncul di laut. 
4. Daynight = untuk menjelaskan terkait kapan waktu satelit menangkap atau mendeteksi adanya sebuah titik. Kolom ini dibagi menjadi dua yaitu D sebagai Day atau waktu siang dan N untuk Night sebagai waktu malam.
5. Satellite = untuk menjelaskan dalam mendeteksi titik panas atau api itu menggunakan satelit apa. SNPP merupakan satu dari banyaknya jenis satelit dalam mendeteksi titik api.

## Deskripsi Tren Titik Panas Harian ##

In [15]:
yearly = (
    lf.with_columns(pl.col("acq_date").str.to_date().alias("year"))
    .group_by("year")
    .agg(pl.len().alias("count_of_detections"))
    .sort("year")
    .collect()
)
yearly


year,count_of_detections
date,u32
2016-01-01,765
2016-01-02,960
2016-01-03,1623
2016-01-04,879
2016-01-05,689
…,…
2026-06-26,333
2026-06-27,385
2026-06-28,361


## PENJELASAN ##
1. Pada cell ini fokus pada rentang tanggal berapa banyak titik api yang terdeteksi setiap tanggalnya. Dari output bisa dilihat, bahwa terdapat satu tanggal yang memiliki lonjakan yang tinggi dari tanggal sebelum maupun sesudahnya, yaitu tanggal 3 Januari 2016. Ketika ditelusuri dalam pencarian berita, pada rentang tahun tersebut memang sedang terjadi kemarau panjang yang mengakibatkan kekeringan jangka panjang. Sumber https://www.nasa.gov/centers-and-facilities/goddard/severe-2015-indonesian-fire-season-linked-to-el-ni%C3%B1o-drought/   